# Square piston-tip-tilt basis validation

Visual inspection of the `SquarePTTZonalBasis` support, orientation and command ordering on a rectangular grid. Each modal value is dimensionless; DM commands supply the OPD scale in metres.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import ActuatorGrid, DeformableMirror, Grid, SquarePTTZonalBasis

In [ ]:
grid = Grid(nx=120, ny=80, dx=0.01, dy=0.01)
actuators = ActuatorGrid(n_actuators_x=2, n_actuators_y=2, pitch=0.5)
basis = SquarePTTZonalBasis(
    actuator_grid=actuators,
    pixel_grid=grid,
    influence_width=0.24,
)
matrix = basis.build_command_matrix()
n_actuators = actuators.n_actuators_x * actuators.n_actuators_y
modes = matrix.T.reshape(3, n_actuators, grid.ny, grid.nx)

print('Command matrix:', tuple(matrix.shape))
print('Modes:', tuple(modes.shape), '= (piston/tip/tilt, actuator, ny, nx)')

In [ ]:
labels = ['Piston', 'Tip (x ramp)', 'Tilt (y ramp)']
fig, axes = plt.subplots(3, n_actuators, figsize=(12, 7), constrained_layout=True)

for family in range(3):
    for actuator in range(n_actuators):
        image = modes[family, actuator]
        ax = axes[family, actuator]
        display = ax.imshow(
            image.cpu(),
            origin='lower',
            cmap='coolwarm',
            vmin=-1 if family else 0,
            vmax=1,
        )
        ax.set_title(f'{labels[family]} — actuator {actuator}')
        ax.set(xlabel='x pixel', ylabel='y pixel')

fig.colorbar(display, ax=axes, shrink=0.8, label='Dimensionless modal value')
plt.show()

Expected visual result: piston is constant inside each square; tip is a horizontal ramp; tilt is a vertical ramp; every mode is exactly zero outside its actuator support.

In [ ]:
dm = DeformableMirror(
    grid=grid,
    actuator_grid=actuators,
    pixel_grid=grid,
    control_basis=basis,
    stroke=1e-6,
)
commands = torch.zeros(basis.n_modes)
commands[0] = 200e-9
commands[n_actuators + 1] = 300e-9
commands[2 * n_actuators + 2] = -250e-9
dm.commands = commands

plt.figure(figsize=(7, 4))
plt.imshow(dm.opd.detach().cpu() * 1e9, origin='lower', cmap='coolwarm')
plt.colorbar(label='OPD (nm)')
plt.title(f'Combined rectangular DM OPD — shape {tuple(dm.opd.shape)}')
plt.xlabel('x pixel')
plt.ylabel('y pixel')
plt.show()